In [ ]:
import os
import re 
import json 
from docx import Document

In [2]:
# Nuevo patrón: número + año + letra final
patron = r"(\d+)\s*/\s*(\d{4})([A-Z])"


In [3]:
def extraer_metadata_desde_texto(texto: str):
    resultados = []
    lineas = texto.split("\n")
    patron = r"(\d{2,4})\s*/\s*(\d{4})([A-Z])"

    for linea in lineas:
        matches = list(re.finditer(patron, linea))
        if not matches:
            continue

        datos = {"pseudonimo": None}

        for match in matches:
            numero, anio, tipo = match.groups()

            if tipo == "C":
                datos["numeroCamara"]  = numero
                datos["anioCamara"]    = anio
            elif tipo == "S":
                datos["numeroSenado"]  = numero
                datos["anioSenado"]    = anio

            if datos["pseudonimo"] is None:
                raw = linea.split(match.group())[0].strip()
                datos["pseudonimo"] = re.sub(r"^\d+\s*", "", raw)

        if datos.get("numeroCamara") or datos.get("numeroSenado"):
            resultados.append(datos)

    return resultados


In [4]:
def procesar_docx_individual(path_docx: str, carpeta_salida: str):
    os.makedirs(carpeta_salida, exist_ok=True)

    doc = Document(path_docx)
    texto_tablas = []
    for table in doc.tables:
        for row in table.rows:
            fila = []
            for cell in row.cells:
                fila.append(cell.text.strip())
            texto_tablas.append(" ".join(fila))

    texto = "\n".join(texto_tablas)

    print("🔍 Primeras líneas del documento extraído:")
    print(texto[:500])

    resultados = extraer_metadata_desde_texto(texto)
    print(f"✅ Total de filas extraídas: {len(resultados)}")

    nombre_base = os.path.splitext(os.path.basename(path_docx))[0]

    if not resultados:
        print("⚠️ No se encontró ninguna fila con metadatos válidos.")
        return

    # Guardar JSONs individuales
    for idx, fila in enumerate(resultados):
        nombre_archivo = f"{nombre_base}_{idx+1}.json"
        ruta_json = os.path.join(carpeta_salida, nombre_archivo)

        with open(ruta_json, "w", encoding="utf-8") as f:
            json.dump(fila, f, indent=4, ensure_ascii=False)

    # Guardar JSON conjunto
    ruta_json_todo = os.path.join(carpeta_salida, f"{nombre_base}_todo.json")
    with open(ruta_json_todo, "w", encoding="utf-8") as f:
        json.dump(resultados, f, indent=4, ensure_ascii=False)

    print(f"📁 Archivos JSON generados en: {carpeta_salida}")


In [5]:
from docx import Document

# Intenta abrir el archivo directamente
ruta = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\agendas.docx"
doc = Document(ruta)
print("✅ Documento cargado correctamente.")


✅ Documento cargado correctamente.


In [6]:
carpeta_docx = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\agendas.docx"
carpeta_salida = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\2021_2022\agenda"

procesar_docx_individual(carpeta_docx, carpeta_salida)

🔍 Primeras líneas del documento extraído:
Total Tema Proyecto No
1 ABANDERAMIENTO DE NAVES 464/2020C
2 ABSTENCIONISMO ESCOLAR 101/2020C
3 ABUSO POLICIAL 411/2020C
4 ACCESO A LA VIVIENDA 054/2020C
5 ACCESO PRIORITARIO VIVIENDA 587/2021C
6 ACCIDENTE EN ANIMAL DOMESTICO 318/2020C
7 ACCIÓN COMUNAL 474/2020C
8 ACCIONES AFIRMATIVAS PARA MUJERES CABEZA DE FAMILIA 498/2020C
9 ACOMPAÑAMIENTO ESTUDIO IDIOMAS 379/2020C
10 ACOSO SEXUAL EN ESPACIO PUBLICO 483/2020C
11 ACTIVIDAD ARTESANAL 490/2020C
12 ACUERDO BANCO EUROPEO 590/2021C
13 ACUERDO ESCAZÚ
✅ Total de filas extraídas: 2245
📁 Archivos JSON generados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\2021_2022\agenda
